In [ ]:
# Full name
NAME = ""
# Institutional email (hm.edu or hmtm.de)
EMAIL = ""

<a href="https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A3_existing_models/10_2_decoding_strategies.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Using Large Models

+ **AI in Culture and Arts - Tech Crash Course**
+ **Date:** 21.05.2026
+ **Author:** Dr. Benedikt Zönnchen

In [ ]:
#@title install dependencies to play sound
%%capture
print('installing fluidsynth...')
!apt-get install fluidsynth > /dev/null
!cp /usr/share/sounds/sf2/FluidR3_GM.sf2 ./font.sf2
print('done!')

In [ ]:
#@title install dependencies to show score in music notation
%%capture
print('installing musescore3...')
!apt-get install musescore3 > /dev/null
print('done!')

In [ ]:
#@title Setup: install required Python packages

%pip install music21
%pip install pyfluidsynth

%pip install matplotlib
%pip install seaborn

%pip install pandas
%pip install numpy
%pip install torch

%pip install otter-grader==5.5.0

In [ ]:
#@title Setup: download assignment files (run this cell)

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # download test files
    import requests, os

    folders = ['tests', 'data', 'models']
    link = "https://api.github.com/repos/aica-wavelab/aica-assignments/contents/A3_existing_models"

    def download(entry, dest):
        if entry.get('type') != 'file' or not entry.get('download_url'):
            return
        r = requests.get(entry['download_url'])
        r.raise_for_status()
        with open(dest, 'wb') as out:
            out.write(r.content)

    for folder in folders:
        os.makedirs(folder, exist_ok=True)
        for f in requests.get(f"{link}/{folder}").json():
            download(f, f"{folder}/{f['name']}")

    for f in requests.get(link).json():
        if f['name'].endswith('.py'):
            download(f, f['name'])

    # Initialize Otter
    import otter
    grader = otter.Notebook(colab=True)
else:
    import otter
    grader = otter.Notebook('10_2_decoding_strategies.ipynb')

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

## 27 Controlling Generation: Decoding Strategies

Every time the transformer predicts the next token, it outputs a **probability distribution** over the entire vocabulary. In principle, all tokens have some (possibly tiny) probability.

🗣 **Remark:** We call this probability distribution, keep in mind that this distribution was learned. It is an estimation.

The question of *how* we pick the next token from this distribution is called the **decoding strategy**. It has a surprisingly large impact on the character of generated melodies:

- **Greedy decoding**: always pick the single most probable token. Deterministic but often repetitive.
- **Temperature sampling**: scale the distribution before sampling. We explored this in notebook 9_8.
- **Top-k sampling**: only consider the *k* most probable tokens. Prevents very unlikely tokens from being picked.
- **Top-p (nucleus) sampling**: consider the smallest set of tokens that together account for at least probability *p*. Adapts to the shape of the distribution.
- **Constrained decoding**: forbid tokens that violate a musical rule (e.g. notes outside a chosen key).

We will implement and compare all of these.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile, glob
import music21 as m21

from encoder import PianoRollEncoder, StringToIntEncoder, TERM_SYMBOL
from files import load_midi_files
from transformer import TransformerDecoder

In [ ]:
# Configure our plotting engine to get nice visualiziations
sns.set_theme(style="whitegrid")
sns.set_context("talk", font_scale=0.8)
sns.set_palette("viridis")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
#@title Load vocabulary and pre-trained model
with zipfile.ZipFile('data/deu_folk_songs.zip', 'r') as z:
    z.extractall('data/deu_folk_songs/')

time_step = 0.5
mid_files = glob.glob('data/deu_folk_songs/**/*.mid', recursive=True)
streams   = load_midi_files(mid_files, time_step=time_step, transpose_to_major=True, max_files=1000)

piano_roll_encoder = PianoRollEncoder(time_step=time_step)
piano_rolls, _     = piano_roll_encoder.encode_streams(streams)
string_to_int      = StringToIntEncoder(piano_rolls)
vocab_size         = len(string_to_int)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if not torch.cuda.is_available():
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu') # for mac gpu
sequence_len = 64

model = TransformerDecoder(
    vocab_size=vocab_size, sequence_len=sequence_len,
    n_embd=32, n_heads=4, n_blocks=2, dropout=0.0
).to(device)

model.load_state_dict(torch.load('models/transformer_model_1000_120.pt', map_location=device))
model.eval()
print('Model ready.')

### 27.1 A Generation Helper

The function below generates a melody using a *sampling function* that you supply. This makes it easy to swap in different decoding strategies without changing the generation loop.

In [ ]:
def generate_melody(model, sequence_len, seed, string_to_int, sample_fn, max_len=120):
    """
    Generate a melody using the provided sample_fn.

    Parameters
    ----------
    sample_fn : callable
        Takes a 1-D logit tensor and returns a single integer token index.
    """
    padded   = [TERM_SYMBOL] * sequence_len + seed
    seed_int = string_to_int.encode_sequence(padded)
    melody   = seed[:]

    model.eval()
    with torch.no_grad():
        while True:
            window  = seed_int[-sequence_len:]
            idx     = torch.tensor([window], dtype=torch.long, device=device)
            logits  = model(idx)[0, -1, :]    # (vocab_size,) — last position

            symbol_int = sample_fn(logits)
            seed_int.append(symbol_int)
            symbol = string_to_int.decode(symbol_int)

            if symbol == TERM_SYMBOL:
                break
            melody.append(symbol)
            if len(melody) >= max_len:
                break

    return melody

SEED = ['60', '_', '_', '62', '_', '64', '_']

### 27.2 Greedy Decoding

The simplest strategy: always pick the token with the **highest probability**. The result is deterministic meaning running twice gives the same melody and the result is often repetitive, because the model keeps returning to its most probable single next note.

In [ ]:
def greedy(logits):
    return logits.argmax().item()

melody_greedy = generate_melody(model, sequence_len, SEED, string_to_int, greedy)
print('Greedy melody:', melody_greedy)
piano_roll_encoder.decode_stream(melody_greedy).show('midi')

### 27.3 Temperature Sampling (recap)

Dividing the logits by a temperature $T$ before applying softmax reshapes the distribution:

- $T < 1$: the distribution becomes **sharper** — the most likely token dominates even more.
- $T > 1$: the distribution becomes **flatter** — less likely tokens get more chance.
- $T = 1$: standard sampling from the model's raw distribution.

In [ ]:
def temperature_sample(logits, temperature=1.0):
    probs = torch.softmax(logits / temperature, dim=-1)
    return torch.multinomial(probs, num_samples=1).item()

mels = []
temps = [0.1, 1.0, 10]
for temp in temps:
    fn = lambda logits, t=temp: temperature_sample(logits, temperature=t)
    mel = generate_melody(model, sequence_len, SEED, string_to_int, fn)
    mels.append(mel)
    print(f'temperature={temp}: {mel[:20]}...')

mel_by_temp = list(zip(temps, mels))

In [ ]:
print(f'temperature {mel_by_temp[0][0]}')
piano_roll_encoder.decode_stream(mel_by_temp[0][1]).show('midi')

In [ ]:
print(f'temperature {mel_by_temp[1][0]}')
piano_roll_encoder.decode_stream(mel_by_temp[1][1]).show('midi')

In [ ]:
print(f'temperature {mel_by_temp[2][0]}')
piano_roll_encoder.decode_stream(mel_by_temp[2][1]).show('midi')

### 27.4 Top-k Sampling

A common criticism of plain temperature sampling is that it still gives *some* probability to every token, including very unlikely or musically wrong ones. **Top-k sampling** fixes this by zeroing out all tokens except the $k$ most probable ones before sampling.

More formally: let $\mathbb{V}^{(k)}$ be the set of the $k$ tokens with the highest logit values. Then:

$$P_{\text{top-k}}(x) = \begin{cases} \text{softmax}(z_x / T) & \text{if } x \in \mathbb{V}^{(k)} \\ 0 & \text{otherwise} \end{cases}$$

Typical values are $k \in \{10, 20, 50\}$.

---

🖍 **Exercise 27.1:** Implement ``top_k_sample``. The function should:
1. Keep only the ``k`` highest logits; replace all others with ``-inf``.
2. Apply softmax (optionally with temperature) and sample one token index.

🗣 **Hint:** ``torch.topk(logits, k)`` returns the top-k values and their indices.

---

In [ ]:
def top_k_sample(logits, k, temperature=1.0):
    """
    Sample from the top-k most probable tokens.

    Parameters
    ----------
    logits      : 1-D torch.Tensor of shape (vocab_size,)
    k           : int — number of tokens to keep
    temperature : float — scaling factor

    Returns
    -------
    int — sampled token index
    """
    ...

In [ ]:
grader.check("q271")

In [ ]:
# Generate melodies with different k values
mels = []
ks = [3, 8, vocab_size]
for k in ks:
    fn  = lambda logits, k_=k: top_k_sample(logits, k=k_, temperature=1.0)
    mel = generate_melody(model, sequence_len, SEED, string_to_int, fn)
    mels.append(mel)
    print(f'k={k:3d}: {mel[:20]}...')

mel_by_k = list(zip(ks, mels))

In [ ]:
print(f'k: {mel_by_k[0][0]}')
piano_roll_encoder.decode_stream(mel_by_k[0][1]).show('midi')

In [ ]:
print(f'k: {mel_by_k[1][0]}')
piano_roll_encoder.decode_stream(mel_by_k[1][1]).show('midi')

In [ ]:
print(f'k: {mel_by_k[2][0]}')
piano_roll_encoder.decode_stream(mel_by_k[2][1]).show('midi')

### 27.5 Top-p (Nucleus) Sampling

Top-k has a fixed cutoff, i.e., always exactly $k$ candidates. But the *right* number of candidates should depend on how confident the model is. When the model is very sure, even a large $k$ might include terrible options; when the model is uncertain, a small $k$ might exclude good ones.

**Top-p sampling** (also called **nucleus sampling**) adapts the cutoff: sort tokens by descending probability, and keep the smallest set of tokens whose *cumulative* probability reaches $p$:

$$\mathbb{V}^{(p)} = \text{smallest } \mathbb{V} \subseteq \mathbb{V}_{\text{all}} \text{ such that } \sum_{x \in \mathbb{V}} P(x) \geq p$$

Typical values are $p \in \{0.8, 0.9, 0.95\}$.

In [ ]:
def top_p_sample(logits, p, temperature=1.0):
    """
    Nucleus (top-p) sampling.

    Parameters
    ----------
    logits      : 1-D torch.Tensor of shape (vocab_size,)
    p           : float in (0, 1] — cumulative probability threshold
    temperature : float

    Returns
    -------
    int — sampled token index
    """
    # BEGIN SOLUTION
    # Step 1: sort by descending probability
    sorted_logits, sorted_indices = torch.sort(logits / temperature, descending=True)
    sorted_probs = torch.softmax(sorted_logits, dim=-1)

    # Step 2: compute cumulative probabilities
    cumulative_probs = torch.cumsum(sorted_probs, dim=0)

    # Step 3: remove tokens once cumulative probability exceeds p
    # We shift by one so that the token that *crosses* the threshold is still kept
    remove_mask = cumulative_probs > p
    remove_mask[1:] = remove_mask[:-1].clone()
    remove_mask[0]  = False                          # always keep at least one token

    # Step 4: apply mask (set removed logits to -inf) and sample
    sorted_logits[remove_mask] = float('-inf')
    probs = torch.softmax(sorted_logits, dim=-1)
    sampled_rank = torch.multinomial(probs, num_samples=1).item()
    return sorted_indices[sampled_rank].item()
    # END SOLUTION

In [ ]:
mel_p09 = generate_melody(model, sequence_len, SEED, string_to_int,
                          lambda l: top_p_sample(l, p=0.9, temperature=1.0))
print('top-p=0.9 melody:', mel_p09[:25])
piano_roll_encoder.decode_stream(mel_p09).show('midi')

### 27.6 Visualising the Distribution

Let us visualise what each strategy actually does to the model's output distribution for a single step.

In [ ]:
#@title plot code (can be ignored)

# Get one logit vector from the model
example_tokens = string_to_int.encode_sequence([TERM_SYMBOL]*sequence_len + SEED)
idx_ex = torch.tensor([example_tokens[-sequence_len:]], dtype=torch.long, device=device)
with torch.no_grad():
    raw_logits = model(idx_ex)[0, -1, :].cpu()

raw_probs   = torch.softmax(raw_logits, dim=-1).numpy()
token_names = [string_to_int.decode(i) for i in range(vocab_size)]
order       = np.argsort(raw_probs)[::-1]

# Build top-k mask (k=6) and top-p mask (p=0.8) for illustration
k_val = 8
p_val = 0.9
topk_set = set(torch.topk(raw_logits, k_val).indices.tolist())

sorted_idx   = np.argsort(raw_probs)[::-1]
cumulative   = np.cumsum(raw_probs[sorted_idx])
nucleus_size = int(np.searchsorted(cumulative, p_val)) + 1
nucleus_set  = set(sorted_idx[:nucleus_size].tolist())

#print(f'orig_idx: {order}')
#print(f'topk_set: {topk_set}')
#print(f'nucleus_set: {nucleus_set}')

fig, ax = plt.subplots(figsize=(14, 4))
colors = []
for orig_idx in order:
    if orig_idx in topk_set and orig_idx in nucleus_set:
        colors.append('#2ca02c')   # both
    elif orig_idx in topk_set:
        colors.append('#1f77b4')   # only top-k
    elif orig_idx in nucleus_set:
        colors.append('#ff7f0e')   # only nucleus
    else:
        colors.append('#d3d3d3')   # excluded by both

ax.set_yscale('log')
ax.bar(range(vocab_size), raw_probs[order], color=colors, alpha=0.8)
ax.set_xticks(range(vocab_size))
ax.set_xticklabels([token_names[i] for i in order], rotation=90, fontsize=8)
ax.set_ylabel('Probability')
ax.set_title(f'Token probabilities — blue: top-k={k_val}, orange: nucleus p={p_val}, green: both (log-scale)')
plt.tight_layout()
plt.show()

### 27.7 Constrained Decoding: Staying in Key

The decoding strategies above control *how many* tokens are eligible at each step. But we can also impose **musical constraints**: for example, forbid any note that is not in the chosen key.

For example, we may want to contrain our melody to the *pentatonic scale* which is a musical scale with only five notes per octave or to *major scale*.

C major uses the notes C, D, E, F, G, A, B — i.e. MIDI pitch classes 0, 2, 4, 5, 7, 9, 11 and the pentatonic scale with C as its root uses C, D, E, G, A - i.e. MIDI pitch classes 0, 2, 4, 7, 9.
By setting the logits of all out-of-key pitch tokens to $-\infty$ before sampling, we guarantee that the model never emits an out-of-key note, regardless of what it would normally prefer.
Of course playing in a major key also changes the probability of the notes played. 

🗣 **Remark:** Before training we transposed each melody in major key to C major and each melody in minor key to A minor. The reason behind this is that C major and A minor share the exact same keys and by transposing all melodies into these scales we hoped to make the task of learning the melodies easier assuming that transposing a whole melody does not change its "color". As a consequence, our model is used to these scales. We can, of course, transpose a generated melody into another key.

In [ ]:
# Precompute a mask: True means the token is ALLOWED
def build_key_mask(string_to_int, allowed_pitch_classes):
    """
    Build a boolean mask (vocab_size,) that is True for tokens that are
    either non-pitch tokens (hold, rest, END) or whose pitch class is
    in allowed_pitch_classes.
    """
    mask = torch.zeros(len(string_to_int), dtype=torch.bool)
    for i in range(len(string_to_int)):
        token = string_to_int.decode(i)
        if token in (TERM_SYMBOL, '_', 'r'):
            mask[i] = True       # always allow structural tokens
        else:
            try:
                midi = int(token)
                if midi % 12 in allowed_pitch_classes:
                    mask[i] = True
            except ValueError:
                mask[i] = True
    return mask

C_MAJOR = {0, 2, 4, 5, 7, 9, 11}    # C D E F G A B
C_MAJOR_PENTATNOIC = {0, 2, 4, 7, 9}    # C D E G A  (pentatnoic major)

mask_c_major = build_key_mask(string_to_int, C_MAJOR)
mask_c_pentatonic_major = build_key_mask(string_to_int, C_MAJOR_PENTATNOIC)

allowed = [string_to_int.decode(i) for i in range(vocab_size) if mask_c_major[i]]
print(f'Tokens allowed in C major: {allowed}')

allowed = [string_to_int.decode(i) for i in range(vocab_size) if mask_c_pentatonic_major[i]]
print(f'Tokens allowed in C major: {allowed}')

In [ ]:
def constrained_sample(logits, key_mask, p=0.9, temperature=1.0):
    """
    Sample a token that is in the allowed set (key_mask == True),
    then apply top-p sampling from those allowed tokens.

    Parameters
    ----------
    logits   : 1-D torch.Tensor (vocab_size,)
    key_mask : 1-D boolean torch.Tensor (vocab_size,) — True = allowed
    """
    # BEGIN SOLUTION
    logits = logits.clone()
    logits[~key_mask] = float('-inf')   # forbid out-of-key tokens
    return top_p_sample(logits, p=p, temperature=temperature)
    # END SOLUTION

In [ ]:
# Generate in C major and in A minor and compare
mel_c = generate_melody(model, sequence_len, SEED, string_to_int,
                        lambda l: constrained_sample(l, mask_c_major, p=0.9))
mel_c_pentatonic = generate_melody(model, sequence_len, SEED, string_to_int,
                        lambda l: constrained_sample(l, mask_c_pentatonic_major, p=0.9))

print('C major melody:', mel_c[:20])
piano_roll_encoder.decode_stream(mel_c).show('midi')

In [ ]:
print('C major melody:', mel_c_pentatonic[:20])
piano_roll_encoder.decode_stream(mel_c_pentatonic).show('midi')

What might also be interesting is the change of the logits / probabilities over time i.e. how do these probabilities change over the generation of one sequnece.
The following function plots for a given melodies the probability distribution for the next token.

In [ ]:
#@title plot code (can be ignored)
def plot_probability_shift(melody, num_top_tokens=100):
    probabilities = []

    for i in range(len(melody)):
        part = melody[:i]
        example_tokens = string_to_int.encode_sequence([TERM_SYMBOL]*sequence_len + part)
        idx_ex = torch.tensor([example_tokens[-sequence_len:]], dtype=torch.long, device=device)
        with torch.no_grad():
            raw_logits = model(idx_ex)[0, -1, :].cpu()
        raw_probs   = torch.softmax(raw_logits, dim=-1).numpy()
        probabilities.append(raw_probs)

    matrix = np.array(probabilities)
    matrix = matrix.T
    top_token_indices = np.argsort(matrix.sum(axis=1))[-num_top_tokens:]
    filtered_matrix = matrix[top_token_indices, :]

    # 4. Generate the Plot
    plt.figure(figsize=(14, 7))

    # We use a perceptually uniform colormap like 'viridis' or 'magma'
    sns.heatmap(
        filtered_matrix, 
        cmap="vlag", 
        xticklabels=True, 
        yticklabels=[string_to_int.decode(t) for t in top_token_indices], # Decode indices back to characters/notes if possible
        cbar_kws={'label': 'Probability'}
    )

    plt.title("Transformer Next-Token Probability Shift Over Time", fontweight='bold')
    plt.xlabel("Sequence Generation Step (Time)")
    plt.ylabel("Token / Pitch Candidate")

    plt.tight_layout()
    plt.show()

Let us try this for two generated melodies. You might find regions of high probability and this might give the melody its "color".

In [ ]:
melody1 = generate_melody(model, sequence_len, [], string_to_int, temperature_sample)
plot_probability_shift(melody1, num_top_tokens=30)

In [ ]:
melody2 = generate_melody(model, sequence_len, [], string_to_int, temperature_sample)
plot_probability_shift(melody2, num_top_tokens=30)

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 27.2:** Sometimes we generate rather short melodies.

1. Why is this the case?
2. How could we combat this problem by manipulating the logits / the probabilities?

---

*Your answer here.*

_Type your answer here, replacing this text._

<!-- END QUESTION -->

### 27.8 Summary

In this notebook we learned about different sampling strategies for a given generative model.

| Strategy | Key idea | Musical effect |
|----------|----------|----------------|
| Greedy | Always pick the most likely token | Deterministic, often repetitive |
| Temperature | Scale logits before softmax | Sharper (conservative) or flatter (creative) |
| Top-k | Keep only k candidates | Prevents very unlikely notes; consistent richness |
| Top-p | Keep cumulative probability ≥ p | Adapts to model confidence; popular in LLMs |
| Constrained | Forbid tokens violating a rule | Enforces musical grammar (key, rhythm, …) |

These strategies can be combined: for instance, constrained top-p sampling with temperature is a common recipe for controlled yet creative generation.

In the next notebook 

<a href="https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A3_existing_models/10_3_transfer_learning.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a> 


we move from *controlling the output* to *changing the model itself*: fine-tuning a pre-trained model on a new musical style.